# Notebook A — Dynamique de croissance bactérienne

**Notebook A — Stage de lycée · Laboratoire de microbiologie quantitative  (version stagiaire)**

Ce notebook explore comment des règles simples au niveau de la cellule individuelle
produisent les courbes de croissance observées en labo.

---

### Comment utiliser ce notebook
- Exécuter chaque cellule dans l'ordre : **Shift + Entrée**
- Lire les questions encadrées `>` et y réfléchir avant de passer à la suite
- Modifier les paramètres et observer ce qui change

> **Environnement recommandé** : [Google Colab](https://colab.research.google.com) — aucune installation nécessaire.
> En local : `pip install numpy matplotlib pandas`
>
> **Durée estimée** : 4 demi-journées (sections A1–A4)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configurer la résolution 
plt.rcParams['figure.dpi'] = 100
# Configurer la police  
plt.rcParams['font.size'] = 12
print("Bibliotheques chargees.")

---
## Section A1 — Croissance exponentielle : le modèle déterministe

Une bactérie se divise en deux. Ses deux filles se divisent en deux. Et ainsi de suite.

Si à chaque pas de temps, chaque cellule a une probabilité $p$ de se diviser,
le nombre moyen de nouvelles divisions est $p \times N$.

**Modèle déterministe** : on ignore les fluctuations et on écrit directement :

$$N(t+1) = N(t) \times (1 + p)$$

C'est une **multiplication répétée par le même facteur** $(1+p)$.

> **Avant de coder** : si $N_0 = 10$ et $p = 0.1$, quel est $N$ après 1 pas ?
> Après 2 pas ? Calculez à la main les 3 premières valeurs.

In [ ]:
# Objectif : simuler la croissance déterministe par multiplication répétée,
# et tracer le résultat.
#
# Paramètres (3 nombres) :
#   N0       — nombre initial de cellules
#   p        — probabilité de division par pas de temps (p << 1)
#   n_steps  — nombre de pas de temps simulés
#
# Structure de données à construire :
#   Ns_det — une LISTE de longueur n_steps+1, qui contient le nombre de
#            cellules à chaque pas de temps. Ns_det[0] = N0.
#            À chaque pas : multiplier le nombre de cellules par (1+p)
#            et ajouter le résultat à la liste.
#
# Graphique : tracer Ns_det en fonction de l'indice (= pas de temps),
# échelle linéaire. Titre + labels des axes.

### Représentation logarithmique

En **échelle logarithmique**, la multiplication répétée par $(1+p)$ se transforme
en une **droite** — ce qui permet d'identifier visuellement la phase de croissance
exponentielle sur vos courbes OD.

Plus la pente est forte, plus les bactéries se divisent vite.

> **Question** : quel est le lien entre la pente de cette droite et le paramètre $p$ ?
> Et avec le **temps de doublement** des bactéries ?

In [ ]:
# Objectif : retracer Ns_det (liste de la cellule précédente) en échelle
# logarithmique, pour observer que la multiplication répétée devient une droite.
#
# Pas de nouvelle structure de données — on réutilise Ns_det.
#
# Graphique : même tracé que la cellule précédente, mais avec
# plt.yscale('log').
#
# Calcul supplémentaire :
#   t_doublement — un nombre, le temps de doublement théorique.
#   Formule : t_doublement = log(2) / log(1 + p)
#   (np.log = logarithme, vu en terminale — pour l'instant on se contente
#    de vérifier graphiquement, sur la courbe log, à quel pas N a doublé
#    par rapport à N0).
#
# Afficher t_doublement et la valeur finale de Ns_det avec print().

---
## Section A2 — Simulation stochastique : le hasard cellule par cellule

Le modèle déterministe donne la **valeur moyenne attendue**.
Mais en réalité, chaque cellule décide de diviser ou non de façon **aléatoire**.

**Règle microscopique** : à chaque pas de temps, pour *chaque* cellule individuellement,
on tire au sort — elle divise avec probabilité $p$, indépendamment de ses voisines.

> **Question** : si le hasard joue un rôle à chaque cellule,
> pourquoi obtient-on une courbe régulière au niveau de la population ?

On va simuler cela directement et observer ce qui se passe.

---

> ⚠️ **Note technique** : cette simulation suppose $p \ll 1$ (probabilité petite par pas de temps).
> C'est l'approximation dite de *τ-leaping*, version discrète de l'algorithme de Gillespie.
> Pour $p = 0.1$, l'approximation est très bonne.

In [ ]:
# Objectif : écrire une fonction qui simule UNE trajectoire stochastique
# (chaque cellule "tire au sort" si elle se divise ce pas-ci), puis la
# comparer à la trajectoire déterministe.
#
# Fonction à définir : simulate_one(N0, p, n_steps)
#   - Entrées  : les 3 mêmes paramètres que la cellule §A1 (N0, p, n_steps)
#   - Sortie   : une LISTE Ns de longueur n_steps+1 (même structure que Ns_det)
#   - Principe : à chaque pas, tirer N nombres aléatoires entre 0 et 1
#                (np.random.random(N)). Le nombre de cellules qui se divisent
#                ce pas-ci = le nombre de tirages inférieurs à p.
#                Ajouter ce nombre de divisions à N, et stocker N dans Ns.
#
# Appeler la fonction une fois → Ns_stoch (une liste, même longueur que Ns_det).
#
# Graphique : tracer Ns_det et Ns_stoch sur la même figure pour comparer.

In [ ]:
# Objectif : lancer 30 simulations stochastiques indépendantes (en appelant
# simulate_one 30 fois), visualiser leur dispersion, et calculer leur moyenne.
#
# Structure de données :
#   all_traj — une LISTE DE LISTES : 30 trajectoires, chacune de longueur
#              n_steps+1 (= 30 appels à simulate_one, stockés dans une liste).
#   mean_traj — un TABLEAU numpy de longueur n_steps+1 : la moyenne, pas de
#              temps par pas de temps, des 30 trajectoires.
#              Astuce : np.mean(all_traj, axis=0) calcule directement cette
#              moyenne colonne par colonne.
#
# Graphique : 2 sous-graphiques côte à côte (plt.subplots(1, 2, ...)) —
#   - tracer les 30 trajectoires en transparence (alpha faible)
#   - superposer Ns_det et mean_traj sur les deux
#   - le graphique de droite en échelle logarithmique

> **Observation** : chaque trajectoire est différente, mais leur **moyenne** suit
> de près la courbe déterministe. En échelle log, les trajectoires restent
> groupées autour d'une droite.
>
> C'est la **loi des grands nombres** : des règles aléatoires au niveau individuel
> produisent un comportement prévisible au niveau de la population.
>
> **À explorer** : relancez la simulation avec $N_0 = 2$ au lieu de $N_0 = 10$.
> Les trajectoires sont-elles plus ou moins dispersées ? Pourquoi ?

---
## Section A3 — Épuisement des nutriments et phase stationnaire

Dans un milieu fermé (tube, microplaque), les nutriments s'épuisent au fur et à mesure
que les bactéries se multiplient. La croissance finit par s'arrêter.

**Idée** : la probabilité de division $p$ n'est plus constante —
elle dépend de la concentration en nutriments $S$ disponibles.

### La courbe de Monod

$$p(S) = p_{\max} \times \frac{S}{K_S + S}$$

| Situation | Valeur de $p(S)$ | Interprétation |
|-----------|:---------------:|----------------|
| $S \gg K_S$ | $\approx p_{\max}$ | Nutriments abondants → croissance maximale |
| $S = K_S$ | $p_{\max}/2$ | $K_S$ fixe l'échelle de saturation |
| $S \ll K_S$ | $\approx 0$ | Nutriments épuisés → croissance arrêtée |

Cette forme est identique à la cinétique enzymatique de Michaelis-Menten.

In [ ]:
# Objectif : visualiser la courbe de Monod p(S) = p_max * S / (Ks + S)
# avant de l'utiliser dans la simulation.
#
# Paramètres :
#   p_max — probabilité de division maximale (nutriments abondants)
#   Ks    — demi-saturation : valeur de S pour laquelle p = p_max / 2
#
# Structure de données :
#   S_vals — un TABLEAU numpy de valeurs de S, de 0 à 200 (np.linspace)
#   p_vals — un TABLEAU numpy de même longueur : p(S) calculé pour
#            chaque valeur de S_vals
#
# Graphique : tracer p_vals en fonction de S_vals. Ajouter deux lignes
# horizontales de repère (p_max et p_max/2) et une ligne verticale à Ks.

In [ ]:
# Objectif : écrire une fonction qui simule la croissance avec épuisement
# des nutriments (modèle de Monod), puis lancer plusieurs trajectoires.
#
# Fonction à définir : simulate_nutrients(N0, S0, p_max, Ks, Y, n_steps)
#   - Entrées  : N0, S0 (quantités initiales de cellules et de nutriments),
#                p_max, Ks (comme cellule précédente), Y (rendement —
#                cellules produites par unité de nutriment consommée),
#                n_steps
#   - Sorties  : DEUX listes Ns et Ss, chacune de longueur n_steps+1
#                (nombre de cellules et quantité de nutriments restants
#                à chaque pas)
#   - Principe : à chaque pas, calculer p = p_max * S / (Ks + S) (Monod),
#                puis comme en §A2 : tirer N nombres aléatoires, compter
#                les divisions. Chaque division consomme 1/Y de nutriment.
#                S ne doit jamais devenir négatif.
#
# Paramètres à fixer : N0_m, S0_m, p_max_m, Ks_m, Y_m, n_m
#
# Graphique : 2 sous-graphiques superposés (partage de l'axe des temps) —
#   lancer 15 trajectoires avec simulate_nutrients, tracer Ns (en haut,
#   échelle log) et Ss (en bas) pour chacune, en transparence.

### Prédiction de la capacité portante

Dans notre modèle, chaque division consomme exactement $1/Y$ unités de nutriments
($Y$ est le **rendement** : nombre de cellules produites par unité de nutriment).

Si on part de $S_0$ nutriments et $N_0$ cellules, quand tous les nutriments sont épuisés :

$$N_{\max} = N_0 + Y \times S_0$$

**Ce n'est pas un paramètre ajusté — c'est une prédiction du modèle.**
La cellule suivante va vérifier cette prédiction numériquement.

In [ ]:
# Objectif : vérifier numériquement la prédiction théorique du modèle
# de Monod : N_max = N0 + Y * S0 (la capacité portante).
#
# Structures de données :
#   N_max_pred — un nombre (la prédiction théorique, calculée directement
#                avec la formule)
#   N_finals   — une LISTE de 50 nombres : pour 50 appels indépendants à
#                simulate_nutrients, la valeur FINALE de Ns (Ns[-1])
#
# Afficher avec print() : N_max_pred, puis la moyenne et l'écart-type de
# N_finals (np.mean, np.std). Ces deux valeurs doivent être proches.

> **Note** : la simulation donne typiquement ~1 % de plus que la prédiction
> $N_{\max} = N_0 + Y \cdot S_0$. La prédiction suppose une coupure nette
> quand $S = 0$, mais dans le modèle discret $p(S) \to 0$ progressivement —
> quelques divisions ont encore lieu quand $S$ est très petit.
> C'est un écart attendu, pas un bug.

---
## Section A4 — Comparaison avec les données expérimentales

### Premier test : la biomasse produite est-elle proportionnelle au glucose ?

Le modèle de Monod prédit $N_{\max} = N_0 + Y \times S_0$,
soit une relation **linéaire** entre biomasse produite et glucose initial,
avec une droite qui passe par l'origine.

On peut le vérifier directement avec vos données de mardi :
pour chaque tube, on calcule $\Delta\text{OD} = \text{OD}_{\max} - \text{OD}_0$.
Si le modèle est juste, les trois points doivent être alignés sur une droite passant par zéro.

Le **rendement** $Y$ est la pente de cette droite :
combien d'unités OD de bactéries sont produites par gramme de glucose consommé.

In [ ]:
# Objectif : à partir de vos 3 valeurs d'OD maximale mesurées mardi (une
# par concentration de glucose), estimer le rendement Y et vérifier la
# relation linéaire biomasse = Y × [glucose].
#
# Données à entrer (3 valeurs chacune, une par concentration) :
#   glucose_gL    — les 3 concentrations en g/L (0.02%, 0.2%, 0.4% → en g/L)
#   OD_initial    — l'OD au temps 0 (un seul nombre, le même pour les 3 tubes)
#   OD_max_mesure — vos 3 valeurs d'OD maximale lues sur les courbes de mardi
#
# Calculs à faire :
#   delta_OD — une LISTE de 3 valeurs : biomasse produite = OD_max - OD_initial
#   Y_vals   — une LISTE de 3 valeurs : pour chaque condition,
#              Y = delta_OD / [glucose]   (un rendement estimé séparément
#              pour chacune des 3 concentrations)
#   Y_fit    — UN seul nombre : la moyenne des 3 valeurs de Y_vals
#
# Graphique : tracer delta_OD (3 points) en fonction de glucose_gL, et
# superposer la droite y = Y_fit * x.
#
# Afficher avec print() : delta_OD, Y_vals, Y_fit, et une estimation du
# rendement en cellules/(g/L) (1 unité d'OD ≈ 2×10⁹ cellules/mL).

### Lecture des données du plate reader
Le plate reader a mesuré l'OD600 de vos cultures toutes les quelques minutes
pendant une nuit entière. On va charger ces données pour
1.  vérifier la biomasse produite en fonction des nutriments pour differentes concentrations de glucose et de lactose.
2. mesurer le taux de croissance dans ces differentes conditions
3. visualiser les courbes de croissance en diauxie.

#### Format du fichier BioTek Gen5

Même format que dans le notebook d'induction : deux blocs TSV (OD + GFP),
avec pour chaque bloc le nom du canal en première ligne.
La fonction `parse_biotek` ci-dessous gère ce format.

> **Note** : si vous avez mesuré des courbes de croissance manuellement (OD au spectro),
> vous pouvez charger un CSV simple avec `pd.read_csv()` :
> colonne `temps_h` (heures) + une colonne par tube.

In [56]:
import pandas as pd

def parse_biotek(filepath):
    # Lit un fichier plate reader BioTek Gen5 (format TSV multi-blocs).
    # Retourne un dict {nom_canal : DataFrame}
    # Chaque DataFrame : colonne 'temps_h' (heures decimales) + colonnes A1...H12.
    with open(filepath, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()
    block_starts = []
    for i, line in enumerate(lines):
        s = line.strip()
        if s and chr(9) not in s and not s[0].isdigit():
            block_starts.append(i)
    result = {}
    for b_idx, b_start in enumerate(block_starts):
        canal_name = lines[b_start].strip()
        if b_start + 2 >= len(lines):
            continue
        header = lines[b_start + 2].strip().split(chr(9))
        b_end = block_starts[b_idx+1] if b_idx+1 < len(block_starts) else len(lines)
        rows = []
        for line in lines[b_start + 3 : b_end]:
            parts = line.strip().split(chr(9))
            if len(parts) < 3:
                continue
            t_str = parts[0]
            if not t_str or t_str == '0:00:00':
                continue
            t_parts = t_str.split(':')
            if len(t_parts) != 3:
                continue
            try:
                h, m, s = int(t_parts[0]), int(t_parts[1]), int(t_parts[2])
            except ValueError:
                continue
            rows.append([h + m/60 + s/3600] + parts[2:])
        if not rows:
            continue
        puits = header[2:]
        n_cols = len(rows[0]) - 1
        cols = ['temps_h'] + puits[:n_cols]
        df = pd.DataFrame(rows, columns=cols)
        for col in df.columns[1:]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        result[canal_name] = df
    return result


# ══════════════════════════════════════════════════════════════════════════════════
# Pour charger VOS données BioTek, décommentez :
# ══════════════════════════════════════════════════════════════════════════════════
# data  = parse_biotek('votre_diauxie.txt')
# od_df = next(df for name, df in data.items() if '600' in name)
#
# Pour un CSV manuel (temps + une colonne par tube) :
# od_df = pd.read_csv('croissance_manuelle.csv').rename(columns={'temps': 'temps_h'})

# ── Données synthétiques diauxie — À REMPLACER par vos données réelles ─────────
np.random.seed(3)

def generer_diauxie():
    # Simule les courbes BioTek d'une expérience diauxie overnight.
    # Puits A1-A3 : glucose seul (3 concentrations)
    # Puits B1-B2 : glucose + lactose (diauxie)
    # Puits H12   : milieu seul (blanc)
    n_pts = 72
    temps = np.linspace(0, 14, n_pts)

    def logistic(t, K, t_mid, mu):
        return K / (1 + np.exp(-mu * (t - t_mid)))

    od = {'temps_h': temps}
    # Glucose seul : 3 concentrations → 3 capacités portantes différentes
    for label, K in [('A1', 0.35), ('A2', 0.80), ('A3', 1.40)]:
        vals = 0.05 + logistic(temps, K, K*3.8, 1.8)
        od[label] = np.clip(vals + np.random.normal(0, 0.008, n_pts), 0, None)
    # Glucose + Lactose : diauxie (deux phases avec creux intermédiaire)
    for label in ['B1', 'B2']:
        phase1 = logistic(temps, 0.45, 2.5, 2.2)
        # Phase lactose démarre après induction lacZ (~5h de lag total)
        phase2 = np.array([logistic(t, 0.42, 9.0, 1.6) if t > 5.0 else 0.0
                           for t in temps])
        vals = 0.05 + phase1 + phase2
        od[label] = np.clip(vals + np.random.normal(0, 0.010, n_pts), 0, None)
    # Blanc
    od['H12'] = np.clip(0.082 + np.random.normal(0, 0.003, n_pts), 0, None)

    return pd.DataFrame(od)

od_df = generer_diauxie()
print("Donnees chargees.")
print(f"  {len(od_df)} points, {len(od_df.columns)-1} puits")
print(f"  Duree : {od_df['temps_h'].min():.1f} – {od_df['temps_h'].max():.1f} h")
print(od_df[['temps_h', 'A1', 'A2', 'B1', 'H12']].head(3).to_string(index=False))

Donnees chargees.
  72 points, 6 puits
  Duree : 0.0 – 14.0 h
 temps_h       A1       A2       B1      H12
0.000000 0.093580 0.052112 0.062726 0.081981
0.197183 0.093799 0.068923 0.053059 0.086503
0.394366 0.105564 0.060650 0.058268 0.079390


### Relation biomasse / nutriments — vérification sur le plate reader

Le plate reader nous donne l'OD600 de **beaucoup** de conditions simultanément.
On va d'abord tracer toutes les courbes de croissance, extraire l'OD maximale
de chaque puits, puis vérifier que $\Delta\text{OD} \propto [\text{glucose}]$.

On pourra aussi comparer le rendement $Y$ mesuré ici avec celui obtenu mardi
à la main — et l'étendre aux puits **lactose seul** si le layout le permet.

In [ ]:
# Objectif : tracer les courbes de croissance du plate reader (corrigées
# du blanc) et estimer le rendement Y à partir de ces données — pour le
# comparer au Y mesuré "à la main" mardi (cellule précédente).
#
# Layout à adapter à votre plaque réelle :
#   puits_glu — LISTE des noms de puits contenant "glucose seul"
#               (ex. ['A1', 'A2', 'A3'], dans l'ordre croissant de concentration)
#   puits_lac — LISTE des puits "lactose seul" (peut être vide [])
#   blanc     — UN nom de puits (string), milieu sans bactéries
#   glu_gL, lac_gL — LISTES des concentrations en g/L correspondantes
#
# Calculs :
#   od_blanc      — UN nombre : moyenne de la colonne `blanc` dans od_df
#   delta_glu_pr  — LISTE : pour chaque puits de puits_glu,
#                   (OD_max - OD_initial) une fois le blanc soustrait
#   Y_vals_glu_pr — LISTE : Y = delta_OD / [glucose] pour chaque puits
#   Y_fit_pr      — UN nombre : moyenne de Y_vals_glu_pr
#
# Graphiques :
#   1. Courbes de croissance od_df['temps_h'] vs (od_df[puit] - od_blanc),
#      une courbe par puits de puits_glu (+ puits_lac si non vide)
#   2. delta_glu_pr en fonction de glu_gL, comparé aux points de mardi
#      (delta_OD) et à la droite y = Y_fit_pr * x
#
# Afficher : Y_fit_pr et le comparer à Y_fit (mardi).

### Comparaison avec le modèle de Monod

Maintenant que les données sont chargées, comparons directement la forme
d'une courbe **glucose seul** avec la simulation de Monod.
Si le modèle est cohérent, les deux courbes normalisées doivent se superposer :
même transition exponentielle → stationnaire.

In [ ]:
# Objectif : comparer une courbe de croissance réelle (glucose seul,
# normalisée entre 0 et 1) à la simulation de Monod (également normalisée).
#
# Structures de données :
#   col_ref — UN nom de puits (string) parmi puits_glu, choisi comme référence
#   OD_ref  — une SÉRIE pandas : od_df[col_ref] - od_blanc
#   OD_norm — OD_ref normalisée entre 0 et 1 :
#             (OD_ref - OD_ref.min()) / (OD_ref.max() - OD_ref.min())
#
#   Ns_c    — résultat de simulate_nutrients(N0_m, S0_m, p_max_m, Ks_m, Y_m, n_m)
#             (ne garder que la première sortie : le nombre de cellules)
#   Ns_norm — Ns_c normalisé entre 0 et 1 (même formule que OD_norm,
#             converti d'abord en tableau numpy avec np.array)
#   t_sim   — un TABLEAU numpy de temps, de même longueur que Ns_norm,
#             allant de od_df['temps_h'].min() à .max() (np.linspace)
#
# Graphique : tracer OD_norm vs od_df['temps_h'], et Ns_norm vs t_sim,
# sur la même figure.
#
# Discuter (en commentaire ou à l'oral) : les deux courbes ont-elles la
# même forme ? Quelle phase du modèle correspond à la phase exponentielle ?

### Taux de croissance $\mu$

En phase exponentielle, l'OD double à intervalles réguliers.
La vitesse de doublement — le **taux de croissance** $\mu$ (en h⁻¹) — dépend-elle
de la concentration en glucose ?

En échelle logarithmique, la phase exponentielle apparaît comme une **droite**,
dont la pente est exactement $\mu$.

> **Note** : cette mesure utilise la fonction logarithme (`np.log`),
> étudiée en terminale. Retenez simplement : *pente plus forte = croissance plus rapide*.

In [ ]:
# Objectif : mesurer le taux de croissance µ de chaque puits "glucose seul",
# en ajustant une droite sur log(OD) pendant la phase exponentielle.
#
# Bornes de la phase exponentielle (à ajuster si besoin) :
#   OD_min_exp, OD_max_exp — deux nombres définissant la plage d'OD à utiliser
#
# Pour chaque puits de puits_glu :
#   od_corr — la série OD corrigée du blanc (comme cellule §A4 précédente)
#   mask    — un masque booléen : True pour les points où
#             OD_min_exp < od_corr < OD_max_exp
#   t_exp, od_exp — les temps et OD correspondant au masque
#   coeffs  — résultat de np.polyfit(t_exp, np.log(od_exp), 1) :
#             un ajustement linéaire de log(OD) en fonction du temps.
#             coeffs[0] est la pente = µ (taux de croissance, en h⁻¹).
#             (np.log = logarithme, np.exp = son inverse — étudiés en terminale)
#
# Structures de données à accumuler :
#   mu_vals  — LISTE des valeurs de µ, une par puits
#   puit_ok  — LISTE des noms de puits correspondants (certains puits
#              peuvent être ignorés s'il n'y a pas assez de points)
#
# Graphiques : 2 sous-graphiques —
#   1. OD corrigé vs temps, avec la droite ajustée superposée (en log,
#      via np.exp(coeffs[1] + coeffs[0]*t_exp))
#   2. log(OD) vs temps — la pente y est directement visible
#
# Afficher pour chaque puits : µ et le temps de doublement correspondant
# (0.693 / µ converti en minutes).

### La diauxie : croissance sur un mélange de nutriments

On va superposer les courbes de croissance **glucose seul** et **glucose + lactose**
pour identifier la pause caractéristique de la diauxie.

#### Diauxie — rappel

Quand *E. coli* croît sur glucose + lactose, elle consomme d'abord le glucose
(source préférée), puis marque une **pause** le temps d'induire les enzymes du
lactose (lacZ, lacY), avant de repartir sur le lactose. Cette double croissance
donne son nom à la *diauxie* (du grec *deux fois*) — et explique pourquoi
l'opéron *lac* est régulé !

> **À observer** : la hauteur du premier plateau dépend-elle de [glucose] ?
> La hauteur finale dépend-elle de [lactose] ?

In [ ]:
# Objectif : comparer les puits "glucose + lactose" (diauxie) aux puits
# "glucose seul", en linéaire et en échelle log.
#
# Donnée à fournir :
#   puits_diauxie — LISTE des noms de puits "glucose + lactose"
#
# Graphiques : 2 sous-graphiques —
#   1. Échelle linéaire : superposer od_df[puit] - od_blanc pour les
#      puits de puits_glu (en pointillés) et puits_diauxie (en trait plein)
#   2. Échelle log : pour les puits_diauxie uniquement, tracer
#      log(od_corr) en fonction du temps (ne garder que les points où
#      od_corr > 0.01, sinon log n'est pas défini)
#
# Discuter :
#   - En linéaire, voit-on deux phases de croissance distinctes ?
#   - En log, le creux correspond à la pause d'induction de lacZ.
#     Pendant cette pause, que fait la bactérie ?
#   - L'OD finale (glu+lac) est-elle plus élevée que (glu seul) ?
#     Est-ce cohérent avec N_max = N0 + Y*(S_glu + S_lac) ?

---
## Pour aller plus loin

1. **Modifier les paramètres** : que se passe-t-il biologiquement si `p_max` augmente ?
   Si `Ks` diminue ? Lequel des deux est plus facile à modifier expérimentalement ?

2. **Deux souches** : simuler deux populations avec des `p_max` différents partageant
   le même milieu (même réservoir $S$). Laquelle domine en fin de culture ? Pourquoi ?

3. **Temps de doublement** : à partir de vos données expérimentales, estimez le
   temps de doublement en phase exponentielle.
   *Indice* : c'est la pente de la droite en échelle log.

4. **Variabilité** : comparez la dispersion des trajectoires entre la phase
   exponentielle (N petit) et la phase stationnaire (N grand). Pourquoi change-t-elle ?